In [3]:
import pandas as pd

df = pd.read_csv('../data/faers_raw.csv', low_memory=False)

# Basic checks
print(df.shape)
print(df.isnull().sum())

# Rename columns to match project
df = df.rename(columns={
    'suspect_drug': 'drugname',
    'primary_reaction': 'reaction'
})

# Check columns
print(df.columns)

# Standardise drug names
df['drugname'] = df['drugname'].astype(str).str.upper().str.strip()

# Handle reactions (IMPORTANT: explode if multiple)
df['reaction'] = df['reaction'].astype(str).str.upper().str.strip()

# Create outcome code (FIXED)
def map_outcome(row):
    if row.get('is_fatal', 0) == 1:
        return 'DE'
    elif row.get('is_hospitalized', 0) == 1:
        return 'HO'
    elif row.get('is_life_threat', 0) == 1:
        return 'LT'
    elif row.get('is_disabling', 0) == 1:
        return 'DS'
    else:
        return 'OT'

df['outc_cod'] = df.apply(map_outcome, axis=1)

# Map outcome labels
outcome_map = {
    'DE': 'Death',
    'HO': 'Hospitalised',
    'LT': 'Life-threatening',
    'DS': 'Disabled',
    'CA': 'Congenital anomaly',
    'OT': 'Other'
}
df['outcome_label'] = df['outc_cod'].map(outcome_map)

# Fix age column (IMPORTANT)
df['age'] = pd.to_numeric(df['patient_age_years'], errors='coerce')

# Date processing
df['report_date'] = pd.to_datetime(df['receive_date'], errors='coerce')

# Drop duplicates (FIXED)
df = df.drop_duplicates(subset=['report_id', 'drugname', 'reaction'])

# Handle missing
df = df.dropna(subset=['drugname', 'reaction', 'outc_cod'])

# Filter serious
serious = df[df['outc_cod'].isin(['DE', 'HO', 'LT'])]

# Save files
df.to_csv('../data/faers_clean.csv', index=False)
serious.to_csv('../data/faers_serious.csv', index=False)

print("Cleaning complete")
print("Clean shape:", df.shape)
print("Serious shape:", serious.shape)

(528000, 30)
report_id                   0
receive_date                0
year                        0
month                       0
quarter                     0
serious                     0
serious_flags          301106
is_fatal                    0
is_hospitalized             0
is_life_threat              0
is_disabling                0
reactions                   0
primary_reaction            0
reaction_outcomes           0
patient_recovered           0
num_reactions               0
suspect_drug                0
brand_name                  0
drug_route                  0
drug_indication             0
manufacturer                0
pharm_class                 0
num_drugs                   0
drug_count_category         0
patient_age_years      151509
age_group                   0
patient_sex                 0
patient_weight_kg      379923
country                     0
report_age_days             0
dtype: int64
Index(['report_id', 'receive_date', 'year', 'month', 'quarter', 'serious',